# LeetCode 510 — Inorder Successor in BST II bằng Python

Link đề: https://leetcode.com/problems/inorder-successor-in-bst-ii/ · Độ khó: **Medium** · Chủ đề: Tree, Binary Search Tree, Binary Tree, Parent Pointer

Skeleton LeetCode:

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None
        self.parent = None
"""

class Solution:
    def inorderSuccessor(self, node: 'Node') -> 'Optional[Node]':
        ...
```

Ghi chú: đây là bài **Premium**. Đề và các ví dụ trong notebook được viết theo hiểu biết về bài; mọi đáp án đã được kiểm chứng bằng code (đối chiếu 4 cách và một oracle dựa trên danh sách giá trị đã sắp xếp), nhưng nếu bạn có tài khoản Premium thì nên đối chiếu lại đề gốc. Notebook tự dựng lớp `Node` (có con trỏ `parent`) và hàm dựng cây từ list kiểu LeetCode; cách LeetCode thật sự truyền đầu vào (cây + giá trị của `node`) được mô phỏng bằng hàm `build_bst` và `find`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho **một nút** `node` trong cây nhị phân tìm kiếm (BST) — **không có gốc** nhưng mỗi nút có con trỏ `parent` — hãy trả về nút **kế tiếp theo thứ tự giữa (inorder successor)**, tức nút có khóa **nhỏ nhất lớn hơn** `node.val`; không có thì trả về `None`.
2. Nắm các công cụ Python: định nghĩa `class Node` có `parent`, dựng cây từ list kiểu LeetCode, duyệt inorder (đệ quy và dùng ngăn xếp), so sánh **danh tính** bằng `is`, vòng `while` leo lên cha, `bisect`.
3. Đi từ **leo lên gốc rồi duyệt inorder toàn cây** (`O(n)`) → **chia trường hợp dùng con trỏ `parent`** (`O(h)`, không cần so giá trị) → **leo lên gốc rồi tìm kiếm BST bằng giá trị** (`O(h)`) → **duyệt inorder lặp dừng sớm**.
4. Hiểu mấu chốt: successor chỉ có **hai trường hợp**: (a) có con phải → nút **trái nhất** của cây con phải; (b) không có con phải → leo lên cho tới khi ta đi lên **từ phía con trái** → cha đó chính là successor.
5. Tránh các lỗi kinh điển: chỉ xử lý trường hợp có con phải, trả về `node.parent` bất kể nhánh, quên kiểm tra `None` khi leo qua gốc, trả về **giá trị** thay vì **nút**, và nhầm `node.right` với "nhỏ nhất của cây con phải".

## 2. Đề bài & ví dụ

Cho một `node` trong một **cây nhị phân tìm kiếm (BST)**, hãy trả về **inorder successor** của nút đó trong BST. Nếu `node` không có successor thì trả về `null` (trong Python là `None`).

Successor của `node` là nút có **khóa nhỏ nhất lớn hơn `node.val`**.

Bạn **có quyền truy cập trực tiếp vào `node` nhưng không có gốc của cây**. Mỗi nút có một tham chiếu tới **nút cha** của nó. Định nghĩa của `Node`:

```python
class Node:
    val: int
    left: Node
    right: Node
    parent: Node
```

**Ví dụ 1**

- Input: `tree = [2,1,3]`, `node = 1`
- Output: `2`
- Giải thích: successor của nút `1` là nút `2`. Lưu ý cả `node` lẫn giá trị trả về đều có kiểu `Node`.

**Ví dụ 2**

- Input: `tree = [5,3,6,2,4,null,null,1]`, `node = 6`
- Output: `null`
- Giải thích: nút `6` không có successor (nó là nút lớn nhất).

**Ràng buộc:**

- Số nút trong cây nằm trong khoảng `[1, 10^4]`.
- `-10^5 <= Node.val <= 10^5`
- Mọi nút có **giá trị khác nhau**.

**Follow up:** có thể giải mà **không nhìn vào giá trị** của bất kỳ nút nào không? (Đáp án: được — đó là cách 2 của notebook.)

In [1]:
from typing import Optional, List
from collections import deque
import bisect
import random
import sys


class Node:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None
        self.parent = None

    def __repr__(self):                 # chỉ để in cho dễ nhìn (LeetCode không cần)
        return f"Node({self.val})"


def build_bst(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng) và gán luôn parent."""
    if not values or values[0] is None:
        return None
    root = Node(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        cur = queue.popleft()
        if i < len(values) and values[i] is not None:
            cur.left = Node(values[i])
            cur.left.parent = cur
            queue.append(cur.left)
        i += 1
        if i < len(values) and values[i] is not None:
            cur.right = Node(values[i])
            cur.right.parent = cur
            queue.append(cur.right)
        i += 1
    return root


def find(root, val):
    """Tìm nút có giá trị val (mô phỏng việc LeetCode đưa cho ta đúng nút `node`)."""
    queue = deque([root])
    while queue:
        cur = queue.popleft()
        if cur is None:
            continue
        if cur.val == val:
            return cur
        queue.append(cur.left)
        queue.append(cur.right)
    return None


def tree_to_list(root):
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        cur = queue.popleft()
        if cur is None:
            out.append(None)
        else:
            out.append(cur.val)
            queue.append(cur.left)
            queue.append(cur.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([2, 1, 3], 1, 2),                          # (tree, node, kỳ vọng: giá trị successor hoặc None)
    ([5, 3, 6, 2, 4, None, None, 1], 6, None),
]
for vals, node_val, expected in examples:
    root = build_bst(vals)
    assert tree_to_list(root) == vals
    node = find(root, node_val)
    print(vals, "node =", node, "-> kỳ vọng", expected, "| cha của node:", node.parent)

[2, 1, 3] node = Node(1) -> kỳ vọng 2 | cha của node: Node(2)
[5, 3, 6, 2, 4, None, None, 1] node = Node(6) -> kỳ vọng None | cha của node: Node(5)


## 3. Kiến thức nền

### 3.1. Cây nhị phân tìm kiếm (BST) và thứ tự giữa (inorder)

Trong một **BST**, với mọi nút `x`: mọi giá trị ở **cây con trái** đều `< x.val` và mọi giá trị ở **cây con phải** đều `> x.val`.

**Inorder** là cách duyệt "trái → nút → phải". Điều thú vị: duyệt inorder một BST cho ra các giá trị **tăng dần**. Vậy "successor theo inorder" chính là **số lớn hơn kế tiếp** trong dãy đã sắp xếp. Hãy kiểm chứng:

In [2]:
def inorder_recursive(node, out):
    if node is None:
        return
    inorder_recursive(node.left, out)
    out.append(node.val)
    inorder_recursive(node.right, out)

root = build_bst([5, 3, 6, 2, 4, None, None, 1])
vals = []
inorder_recursive(root, vals)
print("inorder:", vals)
assert vals == sorted(vals)

inorder: [1, 2, 3, 4, 5, 6]


Cây trên có dạng:

```
        5
       / \
      3   6
     / \
    2   4
   /
  1
```

Dãy inorder `1, 2, 3, 4, 5, 6`: successor của `3` là `4`, của `4` là `5`, của `5` là `6`, của `6` là "không có".

### 3.2. Nút có con trỏ `parent`

Bình thường nút cây chỉ biết **con** của mình. Ở bài này mỗi nút còn nhớ **cha**: `node.parent` (và `None` ở gốc). Nhờ vậy, từ **một nút bất kỳ** ta đi **lên** được tới gốc bằng vòng lặp:

```python
cur = node
while cur.parent is not None:
    cur = cur.parent
```

Hãy thử: leo từ nút `1` lên tới gốc và in đường đi.

In [3]:
node1 = find(root, 1)
path = [node1]
cur = node1
while cur.parent is not None:
    cur = cur.parent
    path.append(cur)
print("đường leo:", path)
print("gốc là:", cur, "| gốc có cha không?", cur.parent)

đường leo: [Node(1), Node(2), Node(3), Node(5)]
gốc là: Node(5) | gốc có cha không? None


### 3.3. Duyệt inorder không đệ quy bằng ngăn xếp

Đệ quy có thể sâu `10^4` tầng với cây lệch. Cách lặp dùng một **ngăn xếp** (`list` với `append`/`pop`):

1. Từ `cur`, đi hết về **bên trái**, đẩy mọi nút gặp trên đường vào ngăn xếp.
2. Lấy nút trên cùng ra → **thăm** nó.
3. Chuyển sang **con phải** của nút vừa thăm và lặp lại.

In [4]:
def inorder_iter(root):
    out, stack, cur = [], [], root
    while cur is not None or stack:
        while cur is not None:          # đi hết về trái
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()               # nút "trái nhất" chưa thăm
        out.append(cur.val)             # thăm
        cur = cur.right                 # sang phải
    return out

print(inorder_iter(root))

[1, 2, 3, 4, 5, 6]


### 3.4. So sánh danh tính: `is` và `==`

Với đối tượng `Node` ta quan tâm "**có phải đúng nút này không**" chứ không phải "giá trị có bằng nhau không". Toán tử `is` so sánh **danh tính** (cùng một đối tượng trong bộ nhớ). Ở bài này giá trị các nút khác nhau nên so giá trị cũng được, nhưng khi leo cây ta thường viết `cur is parent.left` để biết "tôi là con trái hay con phải của cha".

In [5]:
a = Node(1)
b = Node(1)           # cùng giá trị nhưng là hai đối tượng khác nhau
print(a is b, a.val == b.val)
p = Node(5)
p.left = a
a.parent = p
print("a là con trái của p?", a is p.left)
print("a là con phải của p?", a is p.right)

False True
a là con trái của p? True
a là con phải của p? False


### 3.5. `and` ngắn mạch khi leo cây

Khi leo lên, ta hay viết điều kiện `while cur.parent is not None and cur is cur.parent.right`. Thứ tự các vế rất quan trọng: Python **ngắn mạch (short-circuit)** — nếu vế trái sai thì **không** đánh giá vế phải. Nhờ vậy khi `cur.parent` là `None` ta **không** truy cập `cur.parent.right` (sẽ lỗi `AttributeError`).

In [6]:
x = None
print(x is not None and x.val > 3)      # False, không lỗi vì vế phải không được chạy
try:
    print(x.val > 3)
except AttributeError as e:
    print("không ngắn mạch thì lỗi:", e)

False
không ngắn mạch thì lỗi: 'NoneType' object has no attribute 'val'


### 3.6. `bisect` để tìm "phần tử lớn hơn kế tiếp" trong list đã sắp xếp

`bisect.bisect_right(a, v)` trả về chỉ số đầu tiên mà `a[i] > v` trong list **đã sắp xếp** `a`. Ta sẽ dùng nó làm **oracle độc lập** khi test (successor = phần tử đứng ngay sau `v` trong dãy sắp xếp).

In [7]:
a = [1, 2, 3, 4, 5, 6]
for v in (3, 6):
    i = bisect.bisect_right(a, v)
    print(v, "->", a[i] if i < len(a) else None)

3 -> 4
6 -> None


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với cây `[5,3,6,2,4,null,null,1]` (dãy inorder `1,2,3,4,5,6`) hãy tự hỏi successor của từng nút:

- **Nút 3** có con phải là `4`. Mọi giá trị trong cây con phải của `3` đều **lớn hơn 3** và **nhỏ hơn 5** (tổ tiên đã chặn trên). Số nhỏ nhất trong cây con phải là nút **trái nhất**: `4` (không có con trái). → successor = `4`.
- **Nút 2** không có con phải (nó có con trái là `1`). Phải đi **lên**: cha của `2` là `3`, và `2` là **con trái** của `3`. Mọi thứ đã nằm giữa `2` và `3` trong dãy inorder nằm trong cây con phải của `2` (rỗng), nên successor là `3`.
- **Nút 4** không có con phải. Leo lên: `4` là **con phải** của `3` → `3` đã được thăm *trước* `4`, không phải successor, tiếp tục leo; `3` là **con trái** của `5` → dừng, successor = `5`.
- **Nút 6** không có con phải. Leo lên: `6` là con phải của `5` → tiếp tục; `5` là gốc (không có cha) → hết đường → không có successor (`None`).

Tóm lại hai quy tắc:

1. **Có con phải** → đi sang phải **một bước**, rồi đi **trái hết mức**.
2. **Không có con phải** → leo lên **cho tới khi ta là con TRÁI của cha**; cha đó là successor. Nếu leo hết mà không gặp → `None`.

Bạn hãy tự áp dụng hai quy tắc này cho nút `1`, `5` và `6` trước khi xem máy chạy ở các mục sau.

## 5. Cách 1 — Brute Force / baseline: leo lên gốc, duyệt inorder cả cây

**Ý tưởng:**

1. Từ `node` leo lên bằng `parent` tới **gốc**.
2. Duyệt inorder toàn cây, ghi các **nút** (không chỉ giá trị) vào list `order`.
3. Tìm vị trí `i` của `node` trong `order` (so bằng `is`), trả về `order[i + 1]` nếu có, ngược lại `None`.

Cách này đúng hiển nhiên vì "successor theo inorder" **chính là phần tử kế tiếp trong danh sách inorder**.

In [8]:
def successor_brute(node):
    root = node
    while root.parent is not None:                # leo lên gốc
        root = root.parent

    order, stack, cur = [], [], root              # inorder lặp, ghi NÚT
    while cur is not None or stack:
        while cur is not None:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        order.append(cur)
        cur = cur.right

    for i, x in enumerate(order):
        if x is node:
            return order[i + 1] if i + 1 < len(order) else None

for vals, node_val, expected in examples:
    node = find(build_bst(vals), node_val)
    res = successor_brute(node)
    print(vals, node_val, "->", res)
    assert (res.val if res else None) == expected

[2, 1, 3] 1 -> Node(2)
[5, 3, 6, 2, 4, None, None, 1] 6 -> None


**Độ phức tạp.**

- **Time:** `O(n)` — leo `O(h)`, duyệt toàn cây `O(n)`.
- **Space:** `O(n)` cho list `order` (ngăn xếp `O(h)`).

**Nút thắt:** ta duyệt **cả cây** chỉ để tìm một nút kế tiếp. Nhưng successor luôn nằm **rất gần** `node` (trong cây con phải của nó hoặc trên đường lên tổ tiên), nên chỉ cần đi theo chiều cao `h` của cây.

## 6. Ý tưởng thuật toán / bước đột phá

Trong dãy inorder, nút kế tiếp của `node` là nút **đầu tiên được thăm sau khi thăm xong `node`**. Sau khi `node` được thăm, thứ tự "trái → nút → phải" cho ta **hai khả năng**:

**(a) `node` có cây con phải.** Phần "phải" của `node` được duyệt ngay sau nó. Nút đầu tiên được thăm trong cây con phải là nút **trái nhất** của nó.

**(b) `node` không có cây con phải.** Duyệt xong `node` thì cây con gốc `node` đã xong; ta "quay lên" cha.
 - Nếu `node` là **con trái** của cha `p`: ta vừa xong cây con trái của `p`, nút kế tiếp là chính `p`.
 - Nếu `node` là **con phải** của cha: cây con gốc cha cũng đã xong (cha và con trái đã thăm trước, giờ xong phần phải) → quay lên tiếp, lặp lại câu hỏi "tôi là con trái hay con phải của cha mình?".
 - Nếu leo tới gốc (không còn cha) mà vẫn là con phải → `node` là nút **lớn nhất**, không có successor.

Quy tắc này **không dùng giá trị** nào; chỉ dùng hình dạng cây và con trỏ `parent`. Mỗi bước leo hoặc xuống đều đi qua một cạnh nên tổng không quá `O(h)`.

## 7. Các cách tiếp theo

### Cách 2 — Chia trường hợp bằng con trỏ `parent` (cách chuẩn, `O(h)`)

**Giải thích từng đoạn:**

1. Nếu `node.right` tồn tại: `cur = node.right`, rồi `while cur.left: cur = cur.left`, trả về `cur`.
2. Ngược lại: `cur = node`. Trong khi `cur.parent` tồn tại **và** `cur` là con **phải** của cha (`cur is cur.parent.right`): `cur = cur.parent`.
3. Khi dừng, hoặc `cur.parent is None` (đã tới gốc → trả `None`), hoặc `cur` là con trái của cha → trả về `cur.parent`. Cả hai trường hợp gói gọn trong `return cur.parent`.

In [9]:
def successor_parent(node):
    if node.right is not None:                    # trường hợp (a)
        cur = node.right
        while cur.left is not None:
            cur = cur.left
        return cur
    cur = node                                    # trường hợp (b)
    while cur.parent is not None and cur is cur.parent.right:
        cur = cur.parent
    return cur.parent                             # None nếu đã leo tới gốc

for vals, node_val, expected in examples:
    node = find(build_bst(vals), node_val)
    res = successor_parent(node)
    print(vals, node_val, "->", res)
    assert (res.val if res else None) == expected

[2, 1, 3] 1 -> Node(2)
[5, 3, 6, 2, 4, None, None, 1] 6 -> None


In [10]:
class Solution:
    def inorderSuccessor(self, node: 'Node') -> 'Optional[Node]':
        if node.right is not None:
            cur = node.right
            while cur.left is not None:
                cur = cur.left
            return cur
        cur = node
        while cur.parent is not None and cur is cur.parent.right:
            cur = cur.parent
        return cur.parent

for vals, node_val, expected in examples:
    res = Solution().inorderSuccessor(find(build_bst(vals), node_val))
    assert (res.val if res else None) == expected
print("OK")

OK


**Độ phức tạp (cách 2):**

- **Time:** `O(h)` với `h` là chiều cao cây (`h ≤ n`; cây cân bằng thì `h = O(log n)`). Trường hợp (a) đi xuống tối đa `h`, trường hợp (b) đi lên tối đa `h`.
- **Space:** `O(1)`.

Đây cũng là lời giải cho **follow up**: không đọc `val` của nút nào.

### Cách 3 — Leo lên gốc, rồi tìm kiếm BST bằng giá trị

**Ý tưởng khác:** vì đây là BST, successor của giá trị `v` có thể tìm từ **gốc** như khi tra BST:

1. Leo lên gốc bằng `parent`.
2. Đi xuống từ gốc: giữ biến `candidate = None`. Tại nút `cur`:
   - nếu `cur.val > v`: `cur` là **ứng viên** (lớn hơn `v`), ghi `candidate = cur`, rồi sang **trái** để tìm ứng viên nhỏ hơn;
   - ngược lại (`cur.val <= v`): sang **phải**.
3. Khi hết đường, `candidate` là nút nhỏ nhất lớn hơn `v`.

Cách này dùng **giá trị** nên không thỏa follow up, nhưng đúng và cũng `O(h)`.

In [11]:
def successor_search(node):
    root = node
    while root.parent is not None:
        root = root.parent
    v = node.val
    cur, candidate = root, None
    while cur is not None:
        if cur.val > v:
            candidate = cur
            cur = cur.left
        else:
            cur = cur.right
    return candidate

for vals, node_val, expected in examples:
    node = find(build_bst(vals), node_val)
    res = successor_search(node)
    assert (res.val if res else None) == expected
print("OK cách 3 trên ví dụ")

OK cách 3 trên ví dụ


In [12]:
class Solution:
    def inorderSuccessor(self, node: 'Node') -> 'Optional[Node]':
        root = node
        while root.parent is not None:
            root = root.parent
        cur, candidate = root, None
        while cur is not None:
            if cur.val > node.val:
                candidate = cur
                cur = cur.left
            else:
                cur = cur.right
        return candidate

for vals, node_val, expected in examples:
    res = Solution().inorderSuccessor(find(build_bst(vals), node_val))
    assert (res.val if res else None) == expected
print("OK")

OK


**Độ phức tạp (cách 3):** Time `O(h)` (leo `O(h)` + tìm `O(h)`); Space `O(1)`.

### Cách 4 — Duyệt inorder lặp bằng ngăn xếp, **dừng sớm**

**Ý tưởng:** như cách 1 nhưng **không lưu cả danh sách**. Ta duyệt inorder lặp từ gốc và dùng cờ `seen`: khi thăm đúng `node` thì đặt `seen = True`; **nút được thăm ngay sau đó** chính là đáp án → trả về và dừng ngay.

Cách này vẫn `O(n)` trong trường hợp xấu nhưng chỉ cần `O(h)` bộ nhớ và có thể dừng sớm.

In [13]:
def successor_early_stop(node):
    root = node
    while root.parent is not None:
        root = root.parent
    stack, cur, seen = [], root, False
    while cur is not None or stack:
        while cur is not None:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        if seen:                    # nút ngay sau `node` trong inorder
            return cur
        if cur is node:
            seen = True
        cur = cur.right
    return None

for vals, node_val, expected in examples:
    node = find(build_bst(vals), node_val)
    res = successor_early_stop(node)
    assert (res.val if res else None) == expected
print("OK cách 4 trên ví dụ")

OK cách 4 trên ví dụ


In [14]:
class Solution:
    def inorderSuccessor(self, node: 'Node') -> 'Optional[Node]':
        root = node
        while root.parent is not None:
            root = root.parent
        stack, cur, seen = [], root, False
        while cur is not None or stack:
            while cur is not None:
                stack.append(cur)
                cur = cur.left
            cur = stack.pop()
            if seen:
                return cur
            if cur is node:
                seen = True
            cur = cur.right
        return None

for vals, node_val, expected in examples:
    res = Solution().inorderSuccessor(find(build_bst(vals), node_val))
    assert (res.val if res else None) == expected
print("OK")

OK


**Độ phức tạp (cách 4):** Time `O(n)` trường hợp xấu (nhưng thường dừng sớm); Space `O(h)`.

## 8. TRACE từng bước

Ta trace **cách 2** cho từng nhóm nút của cây `[5,3,6,2,4,null,null,1]`. Hàm in ra mỗi quyết định.

| Bước | Ý nghĩa |
|---|---|
| `có con phải` | rơi vào trường hợp (a), rồi in từng bước đi trái |
| `leo` | rơi vào trường hợp (b): in `cur`, `cha`, và `cur` là con trái hay phải |
| kết quả | nút trả về hoặc `None` |

In [15]:
def trace_successor(node):
    print(f"=== successor của {node.val}")
    if node.right is not None:
        cur = node.right
        print(f"  có con phải {cur.val}; đi trái hết mức")
        while cur.left is not None:
            print(f"    {cur.val} còn con trái {cur.left.val} -> đi trái")
            cur = cur.left
        print(f"  -> {cur.val} (trái nhất của cây con phải)")
        return cur
    print("  không có con phải; leo lên")
    cur = node
    while cur.parent is not None:
        side = "PHẢI" if cur is cur.parent.right else "TRÁI"
        print(f"    cur={cur.val}, cha={cur.parent.val}, cur là con {side} của cha")
        if side == "TRÁI":
            print(f"  -> {cur.parent.val} (cha, vì ta đi lên từ nhánh trái)")
            return cur.parent
        cur = cur.parent
    print("  -> None (đã leo tới gốc từ nhánh phải)")
    return None

root = build_bst([5, 3, 6, 2, 4, None, None, 1])
for v in [3, 2, 4, 6, 1]:
    trace_successor(find(root, v))

=== successor của 3
  có con phải 4; đi trái hết mức
  -> 4 (trái nhất của cây con phải)
=== successor của 2
  không có con phải; leo lên
    cur=2, cha=3, cur là con TRÁI của cha
  -> 3 (cha, vì ta đi lên từ nhánh trái)
=== successor của 4
  không có con phải; leo lên
    cur=4, cha=3, cur là con PHẢI của cha
    cur=3, cha=5, cur là con TRÁI của cha
  -> 5 (cha, vì ta đi lên từ nhánh trái)
=== successor của 6
  không có con phải; leo lên
    cur=6, cha=5, cur là con PHẢI của cha
  -> None (đã leo tới gốc từ nhánh phải)
=== successor của 1
  không có con phải; leo lên
    cur=1, cha=2, cur là con TRÁI của cha
  -> 2 (cha, vì ta đi lên từ nhánh trái)


Nhìn lại trace:

- `3`: có con phải `4` → `4` không có con trái → successor `4`.
- `2`: không có con phải; `2` là con **trái** của `3` → dừng ngay, successor `3`.
- `4`: leo qua `3` (là con phải của `3`, tiếp tục) rồi `3` là con trái của `5` → successor `5`.
- `6`: leo qua `5`... `6` là con phải của `5`, rồi `5` không còn cha → `None`.
- `1`: là con trái của `2` → successor `2`.

Ta cũng "xem" toàn bộ chuỗi successor bằng cách bắt đầu từ nút nhỏ nhất và gọi liên tục — ta sẽ nhận lại dãy inorder `1..6`:

In [16]:
def smallest(root):
    cur = root
    while cur.left is not None:
        cur = cur.left
    return cur

cur = smallest(root)
chain = []
while cur is not None:
    chain.append(cur.val)
    cur = successor_parent(cur)
print(chain)
assert chain == [1, 2, 3, 4, 5, 6]

[1, 2, 3, 4, 5, 6]


## 9. Code nộp LeetCode

Cách chuẩn (cũng thỏa follow up): chia trường hợp bằng con trỏ `parent`. Trên LeetCode `Node` đã định nghĩa sẵn; ở đây ta đã tự dựng ở đầu notebook.

In [17]:
class Solution:
    def inorderSuccessor(self, node: 'Node') -> 'Optional[Node]':
        # Trường hợp 1: có con phải -> nút trái nhất của cây con phải
        if node.right:
            cur = node.right
            while cur.left:
                cur = cur.left
            return cur
        # Trường hợp 2: không có con phải -> leo lên tới khi là con trái của cha
        cur = node
        while cur.parent and cur is cur.parent.right:
            cur = cur.parent
        return cur.parent


for vals, node_val, expected in examples:
    res = Solution().inorderSuccessor(find(build_bst(vals), node_val))
    print(vals, node_val, "->", res)
    assert (res.val if res else None) == expected

[2, 1, 3] 1 -> Node(2)
[5, 3, 6, 2, 4, None, None, 1] 6 -> None


## 10. Vì sao thuật toán đúng

**Thứ tự inorder.** Inorder duyệt "trái, nút, phải", nên ngay sau khi thăm `node` ta thăm cây con phải của `node` (nếu có), ngược lại ta thoát khỏi cây con gốc `node` và đi tiếp phần còn lại.

**Trường hợp (a): có con phải.** Nút đầu tiên được thăm trong cây con phải theo inorder là nút **trái nhất** (đi trái hết mức rồi mới thăm). Không có nút nào khác đứng giữa `node` và nút đó trong inorder, vì phần giữa chỉ có thể nằm trong cây con phải của `node` nhưng nút trái nhất đã là nút đầu tiên của cây con ấy.

**Trường hợp (b): không có con phải.** Khi leo từ `cur` lên cha `p`:

- nếu `cur` là con **trái** của `p` thì inorder vừa xong cây con trái của `p`, nút kế tiếp chính là `p` → đáp án;
- nếu `cur` là con **phải** của `p` thì `p` và cây con trái của `p` đã được thăm trước đó; toàn bộ cây con của `p` đã xong, ta lặp lại câu hỏi với `p`.

Nếu leo tới gốc mà luôn là con phải, mọi nút khác đã thăm trước `node` → `node` là nút cuối trong inorder → `None`.

**Thời gian.** Mỗi bước đi (xuống ở (a), lên ở (b)) đi qua một cạnh theo một chiều duy nhất, nên tổng ≤ `h`.

**Cách 3** đúng vì định nghĩa successor là "khóa nhỏ nhất lớn hơn `v`" và tính chất BST: nhìn từ gốc, mọi nút `> v` rẽ **trái** để tìm giá trị nhỏ hơn (nhưng vẫn `> v`), mọi nút `<= v` rẽ **phải**; ứng viên cuối cùng được ghi nhận là nhỏ nhất. **Cách 1 và 4** dựa trực tiếp vào định nghĩa inorder.

## 11. Phân tích Time / Space Complexity

Gọi `h` là chiều cao cây, `n` là số nút.

| Cách | Time | Space |
|---|---|---|
| 1. Leo lên gốc + inorder toàn cây | `O(n)` | `O(n)` |
| 2. Chia trường hợp bằng `parent` | `O(h)` | `O(1)` |
| 3. Leo lên gốc + tìm kiếm BST | `O(h)` | `O(1)` |
| 4. Inorder lặp dừng sớm | `O(n)` xấu nhất | `O(h)` |

Với cây lệch `h = n = 10^4` nên `O(h)` vẫn rất nhỏ. Với cây cân bằng `h ≈ 14`.

## 12. Bộ test

### 12.1. Ví dụ đề và edge case

Ta so sánh theo **danh tính** nút: kết quả phải là **chính nút** mong đợi (hoặc `None`), không chỉ cùng giá trị. Oracle độc lập: lấy mọi nút, sắp xếp theo giá trị rồi `bisect` — không dùng con trỏ `parent` hay inorder.

In [18]:
def all_nodes(root):
    out, stack = [], [root]
    while stack:
        x = stack.pop()
        if x is not None:
            out.append(x)
            stack.append(x.left)
            stack.append(x.right)
    return out

def oracle_successor(node):
    root = node
    while root.parent is not None:
        root = root.parent
    nodes = sorted(all_nodes(root), key=lambda x: x.val)
    keys = [x.val for x in nodes]
    i = bisect.bisect_right(keys, node.val)
    return nodes[i] if i < len(nodes) else None

solvers = {
    "brute": successor_brute,
    "parent": successor_parent,
    "search": successor_search,
    "early_stop": successor_early_stop,
    "leetcode": lambda nd: Solution().inorderSuccessor(nd),
}

def check_all_nodes(vals):
    """Kiểm tra mọi nút của cây: mọi cách cho đúng NÚT mà oracle trả về."""
    root = build_bst(vals)
    for nd in all_nodes(root):
        want = oracle_successor(nd)
        for name, fn in solvers.items():
            got = fn(nd)
            assert got is want, (name, vals, nd.val, got, want)

for vals, node_val, expected in examples:
    check_all_nodes(vals)
print("Ví dụ đề: OK cho mọi nút của mỗi cây")

Ví dụ đề: OK cho mọi nút của mỗi cây


In [19]:
edge_cases = [
    ([1], "một nút duy nhất -> None"),
    ([2, 1], "gốc có một con trái"),
    ([1, None, 2], "gốc có một con phải"),
    ([2, 1, 3], "cây 3 nút"),
    ([5, 3, 6, 2, 4, None, None, 1], "ví dụ 2"),
    ([4, 2, 6, 1, 3, 5, 7], "cây đầy đủ"),
    ([5, 4, None, 3, None, 2, None, 1], "lệch trái"),
    ([1, None, 2, None, 3, None, 4, None, 5], "lệch phải"),
    ([10, 5, 15, None, 7, 12, 20, 6, 8], "cây tự đặt, nhiều dạng nhánh"),
    ([-3, -5, 0, -6, -4], "có số âm"),
]
for vals, note in edge_cases:
    check_all_nodes(vals)
    print("OK:", note)

OK: một nút duy nhất -> None
OK: gốc có một con trái
OK: gốc có một con phải
OK: cây 3 nút
OK: ví dụ 2
OK: cây đầy đủ
OK: lệch trái
OK: lệch phải
OK: cây tự đặt, nhiều dạng nhánh
OK: có số âm


In [20]:
# Kiểm một số đáp án cụ thể bằng tay (giá trị successor)
root = build_bst([5, 3, 6, 2, 4, None, None, 1])
want = {1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: None}
for v, w in want.items():
    for name, fn in solvers.items():
        res = fn(find(root, v))
        assert (res.val if res else None) == w, (name, v)
print("bảng successor của cây ví dụ 2:", want)

bảng successor của cây ví dụ 2: {1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: None}


### 12.2. Phản ví dụ cho các cách làm sai

- **Sai 1 — chỉ xử lý trường hợp có con phải** (quên leo lên).
- **Sai 2 — trả về `node.parent` bất kể nút là con trái hay phải.**
- **Sai 3 — lấy `node.right` mà không đi trái hết mức.**

Ta viết các bản sai và chứng minh bộ test bắt được chúng:

In [21]:
def wrong_only_right(node):
    if node.right:
        cur = node.right
        while cur.left:
            cur = cur.left
        return cur
    return None                                  # SAI: quên trường hợp leo lên

def wrong_parent_only(node):
    if node.right:
        cur = node.right
        while cur.left:
            cur = cur.left
        return cur
    return node.parent                           # SAI: cha có thể nhỏ hơn node

def wrong_right_child_only(node):
    return node.right if node.right else node.parent   # SAI: lấy con phải mà không đi trái hết mức

def fails_on(fn, vals):
    root = build_bst(vals)
    return [nd.val for nd in all_nodes(root) if fn(nd) is not oracle_successor(nd)]

tree = [5, 3, 6, 2, 4, None, None, 1]
for fn in (wrong_only_right, wrong_parent_only, wrong_right_child_only):
    bad = fails_on(fn, tree)
    print(fn.__name__, "sai ở các nút:", bad)
    assert bad

wrong_only_right sai ở các nút: [4, 2, 1]
wrong_parent_only sai ở các nút: [6, 4]
wrong_right_child_only sai ở các nút: [6, 4]


### 12.3. Đối chiếu ngẫu nhiên có seed

Ta sinh **BST ngẫu nhiên** hợp lệ: lấy `n` giá trị **khác nhau** trong `[-10^5, 10^5]`, chèn theo thứ tự ngẫu nhiên vào BST (đồng thời gán `parent`). Sau đó kiểm tra **mọi nút** của mọi cây với mọi cách.

In [22]:
def random_bst(rng, n, lo=-10**5, hi=10**5):
    values = rng.sample(range(lo, hi + 1), n)
    root = Node(values[0])
    for v in values[1:]:
        cur = root
        while True:
            if v < cur.val:
                if cur.left is None:
                    cur.left = Node(v)
                    cur.left.parent = cur
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = Node(v)
                    cur.right.parent = cur
                    break
                cur = cur.right
    return root

rng = random.Random(510)
checked = 0
for _ in range(400):
    n = rng.randint(1, 60)
    rng_range = rng.choice([(-5, 80), (-10**5, 10**5)])
    root = random_bst(rng, n, *rng_range)
    for nd in all_nodes(root):
        want = oracle_successor(nd)
        for name, fn in solvers.items():
            assert fn(nd) is want, (name, nd.val)
        checked += 1
print("đã kiểm", checked, "nút trên 400 BST ngẫu nhiên (5 cách khớp oracle theo danh tính)")

đã kiểm 11508 nút trên 400 BST ngẫu nhiên (5 cách khớp oracle theo danh tính)


In [23]:
# Cây lệch 10^4 nút (tối đa theo ràng buộc): tất cả cách đều là vòng lặp nên không lo tràn ngăn xếp
n = 10000
root = Node(1)
cur = root
for v in range(2, n + 1):
    nxt = Node(v)
    cur.right = nxt
    nxt.parent = cur
    cur = nxt
tail = cur
assert successor_parent(tail) is None and successor_search(tail) is None
# tự đi tới nút 5000 (không dùng BFS `find` cho đỡ chậm)
cur = root
for _ in range(4999):
    cur = cur.right
assert cur.val == 5000
assert successor_parent(cur).val == 5001
assert successor_search(cur).val == 5001
assert successor_early_stop(cur).val == 5001
assert successor_brute(cur).val == 5001
# cây lệch trái: successor của nút sâu nhất là cha của nó
root2 = Node(n)
cur = root2
for v in range(n - 1, 0, -1):
    nxt = Node(v)
    cur.left = nxt
    nxt.parent = cur
    cur = nxt
assert successor_parent(cur).val == 2 and successor_search(cur).val == 2
print("cây lệch 10^4 nút: OK")

cây lệch 10^4 nút: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ xử lý trường hợp có con phải

**Code sai:**

In [24]:
class WrongSolution1:
    def inorderSuccessor(self, node):
        if node.right:
            cur = node.right
            while cur.left:
                cur = cur.left
            return cur
        return None                       # quên: nút không có con phải vẫn có thể có successor (tổ tiên)

root = build_bst([5, 3, 6, 2, 4, None, None, 1])
n2 = find(root, 2)
print("sai :", WrongSolution1().inorderSuccessor(n2), "| đúng:", Solution().inorderSuccessor(n2))

sai : None | đúng: Node(3)


**Vì sao sai:** nút `2` không có con phải nhưng successor là `3` — **tổ tiên**. Ở trường hợp (b) ta phải leo lên bằng `parent`.

**Code đúng:** thêm vòng leo `while cur.parent and cur is cur.parent.right: cur = cur.parent` rồi `return cur.parent`.

### Lỗi 2: Trả về `node.parent` bất kể nút là con trái hay phải

In [25]:
class WrongSolution2:
    def inorderSuccessor(self, node):
        if node.right:
            cur = node.right
            while cur.left:
                cur = cur.left
            return cur
        return node.parent               # SAI nếu node là con PHẢI của cha

n4 = find(root, 4)
print("sai :", WrongSolution2().inorderSuccessor(n4), "(cha của 4 là 3 < 4)")
print("đúng:", Solution().inorderSuccessor(n4))

sai : Node(3) (cha của 4 là 3 < 4)
đúng: Node(5)


**Vì sao sai:** `4` là con **phải** của `3`, cha `3` nhỏ hơn `4` nên đã thăm **trước**. Phải leo tiếp lên tới `5`.

### Lỗi 3: Quên kiểm tra `parent is None` khi leo (lỗi `AttributeError`)

In [26]:
class WrongSolution3:
    def inorderSuccessor(self, node):
        if node.right:
            cur = node.right
            while cur.left:
                cur = cur.left
            return cur
        cur = node
        while cur is cur.parent.right:     # SAI: cur.parent có thể là None (cur là gốc)
            cur = cur.parent
        return cur.parent

n6 = find(root, 6)
try:
    WrongSolution3().inorderSuccessor(n6)
except AttributeError as e:
    print("lỗi:", e)
print("đúng:", Solution().inorderSuccessor(n6))

lỗi: 'NoneType' object has no attribute 'right'
đúng: None


**Vì sao sai:** nút lớn nhất leo tới gốc và `root.parent` là `None`; `None.right` gây lỗi. Phải viết `while cur.parent and cur is cur.parent.right:` (nhờ ngắn mạch, vế phải không chạy khi `cur.parent` là `None`).

### Lỗi 4: Trả về giá trị thay vì nút

In [27]:
class WrongSolution4:
    def inorderSuccessor(self, node):
        res = Solution().inorderSuccessor(node)
        return res.val if res else None      # SAI kiểu trả về: đề yêu cầu Node

res = WrongSolution4().inorderSuccessor(find(root, 3))
print("trả về:", res, "| kiểu:", type(res).__name__, "(đề cần Node)")
print("đúng :", Solution().inorderSuccessor(find(root, 3)), "| kiểu:", type(Solution().inorderSuccessor(find(root, 3))).__name__)

trả về: 4 | kiểu: int (đề cần Node)
đúng : Node(4) | kiểu: Node


**Vì sao sai:** trình chấm so sánh **nút trả về** với nút đúng, không phải giá trị số. Trả `4` (một `int`) thay vì `Node(4)` bị coi là sai kiểu.

### Lỗi 5: Lấy `node.right` mà không đi trái hết mức
Đã minh họa ở mục 12.2 (`wrong_right_child_only`): successor là nút **trái nhất** của cây con phải, không phải `node.right`. Ví dụ `node = 5` trong `[5,3,8,None,None,6,9]`: con phải là `8`, nhưng successor là `6`.

In [28]:
vals = [5, 3, 8, None, None, 6, 9]
r = build_bst(vals)
n5 = find(r, 5)
print("sai (node.right):", n5.right, "| đúng:", Solution().inorderSuccessor(n5))
assert Solution().inorderSuccessor(n5).val == 6

sai (node.right): Node(8) | đúng: Node(6)


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Successor / predecessor trong BST"** → nhớ hai trường hợp: có con (đi một bước rồi trái/phải hết mức) và không có con (leo lên theo `parent` hoặc dò từ gốc).
- Có `parent` → leo lên và hỏi "tôi là con trái hay phải?". Không có `parent` mà có gốc → dò từ gốc, ghi **ứng viên** (bài 285).
- Predecessor đối xứng: đổi `left` ↔ `right`.
- So sánh nút bằng `is`; luôn kiểm tra `cur.parent` **trước** khi truy cập `cur.parent.right` (ngắn mạch).
- Bài này **không cần duyệt cả cây**; mỗi khi thấy BST, hãy xem có làm trong `O(h)` được không.
- Phát biểu "thứ tự giữa của BST = dãy tăng dần" là chìa khóa để chuyển bài về "tìm phần tử lớn hơn kế tiếp".

## 15. Biến thể

1. **Inorder Successor in BST (285):** cho **gốc** và nút `p` (không có `parent`) → dò từ gốc ghi ứng viên (cách 3).
2. **Inorder predecessor:** đối xứng (đổi trái ↔ phải).
3. **Successor của một giá trị `v` không nhất thiết có trong cây:** vẫn dò từ gốc.
4. **`k`-th successor:** gọi successor `k` lần (hoặc dùng iterator BST).
5. **BST Iterator (173):** cài `next()` bằng ngăn xếp.
6. **Cây có giá trị trùng:** cần quy ước (inorder vẫn tăng không ngặt); cách 2 vẫn đúng theo thứ tự inorder.

Ví dụ biến thể 2 (predecessor) và 3 (successor của giá trị bất kỳ):

In [29]:
def predecessor_parent(node):
    if node.left is not None:
        cur = node.left
        while cur.right is not None:
            cur = cur.right
        return cur
    cur = node
    while cur.parent is not None and cur is cur.parent.left:
        cur = cur.parent
    return cur.parent

def successor_of_value(root, v):
    cur, candidate = root, None
    while cur is not None:
        if cur.val > v:
            candidate, cur = cur, cur.left
        else:
            cur = cur.right
    return candidate

root = build_bst([5, 3, 6, 2, 4, None, None, 1])
print("predecessor của 5:", predecessor_parent(find(root, 5)))
print("predecessor của 1:", predecessor_parent(find(root, 1)))
print("successor của giá trị 3.5:", successor_of_value(root, 3.5))
print("successor của giá trị 100:", successor_of_value(root, 100))

predecessor của 5: Node(4)
predecessor của 1: None
successor của giá trị 3.5: Node(4)
successor của giá trị 100: None


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Leo lên gốc + inorder toàn cây | Lấy danh sách inorder, tra vị trí | `O(n)` | `O(n)` | Đối chiếu, dễ đúng |
| 2. Chia trường hợp bằng `parent` | Có con phải / leo lên | `O(h)` | `O(1)` | **Cách chuẩn, thỏa follow up** |
| 3. Leo lên gốc + dò BST | Ghi ứng viên khi `cur.val > v` | `O(h)` | `O(1)` | Khi quen bài 285; dùng giá trị |
| 4. Inorder lặp dừng sớm | Cờ `seen`, trả nút kế tiếp | `O(n)` xấu nhất | `O(h)` | Khi muốn tránh đệ quy, tổng quát cho mọi cây |

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Với cây `[5,3,6,2,4,None,None,1]`, `Solution().inorderSuccessor(find(root, 3))` trả về nút nào? Dự đoán rồi chạy.

In [30]:
root = build_bst([5, 3, 6, 2, 4, None, None, 1])
print(Solution().inorderSuccessor(find(root, 3)))

Node(4)


**Bài 2 (tính tay).** Với cùng cây, ghi list giá trị successor của các nút `1, 2, 3, 4, 5, 6` (theo thứ tự đó, dùng `None` nếu không có) vào `ans2`.

In [31]:
ans2 = None      # TODO: ví dụ [x1, x2, x3, x4, x5, x6]

**Bài 3 (điền TODO).** Viết hàm `leftmost(node)` trả về nút trái nhất (đi `left` hết mức) của cây con gốc `node`. Sau đó dùng nó để viết `successor_with_leftmost(node)` = cách 2.

In [32]:
def leftmost(node):
    # TODO
    pass

def successor_with_leftmost(node):
    # TODO: dùng leftmost; leo lên bằng parent khi không có con phải
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_successor(node)` rồi kiểm tra trên mọi nút của một vài cây.

In [33]:
def my_successor(node):
    # TODO
    pass

**Bài 5 (mini challenge).** Viết `kth_successor(node, k)`: đi `k` lần `successor` và trả về nút đó (hoặc `None` nếu hết). Ví dụ `kth_successor(find(root, 2), 3)` trên cây trên là nút `5` (`2 -> 3 -> 4 -> 5`). Sau đó thử viết `predecessor(node)` (đối xứng).

In [34]:
def kth_successor(node, k):
    # TODO
    pass

def predecessor(node):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: `3` có con phải không? Nếu có, đi trái hết mức từ con phải đó.
- Bài 2: viết ra dãy inorder; successor là phần tử đứng ngay sau.
- Bài 3: một vòng `while node.left`.
- Bài 4: hai trường hợp: có con phải / không có.
- Bài 5: gọi lặp lại `Solution().inorderSuccessor`.

**Mức 2 (vừa):**
- Bài 2: `[2, 3, 4, 5, 6, None]`. Tự kiểm bằng hai quy tắc.
- Bài 3: `while node.left: node = node.left; return node`.
- Bài 4: không có con phải thì leo cho tới khi `cur` là con trái của cha.
- Bài 5: `for _ in range(k): if node is None: return None; node = successor(node)`. Predecessor: đổi `left`/`right`.

**Mức 3 (gần đáp án):**
- Bài 4: `while cur.parent and cur is cur.parent.right: cur = cur.parent; return cur.parent`.
- Bài 5: xem `predecessor_parent` ở mục 15.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [35]:
# Bài 1: successor của 3 là 4 (con phải của 3, không có con trái)
assert Solution().inorderSuccessor(find(root, 3)).val == 4

# Bài 2
ans2 = [2, 3, 4, 5, 6, None]
got = []
for v in [1, 2, 3, 4, 5, 6]:
    r_ = Solution().inorderSuccessor(find(root, v))
    got.append(r_.val if r_ else None)
assert got == ans2
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [36]:
# Bài 3
def leftmost(node):
    while node.left is not None:
        node = node.left
    return node

def successor_with_leftmost(node):
    if node.right is not None:
        return leftmost(node.right)
    cur = node
    while cur.parent is not None and cur is cur.parent.right:
        cur = cur.parent
    return cur.parent

for nd in all_nodes(root):
    assert successor_with_leftmost(nd) is oracle_successor(nd)
assert leftmost(root).val == 1
print("Bài 3 OK")

Bài 3 OK


In [37]:
# Bài 4
def my_successor(node):
    if node.right:
        cur = node.right
        while cur.left:
            cur = cur.left
        return cur
    cur = node
    while cur.parent and cur is cur.parent.right:
        cur = cur.parent
    return cur.parent

rng = random.Random(4)
for _ in range(100):
    rt = random_bst(rng, rng.randint(1, 40))
    for nd in all_nodes(rt):
        assert my_successor(nd) is oracle_successor(nd)
print("Bài 4 OK")

Bài 4 OK


In [38]:
# Bài 5
def kth_successor(node, k):
    for _ in range(k):
        if node is None:
            return None
        node = Solution().inorderSuccessor(node)
    return node

def predecessor(node):
    return predecessor_parent(node)

assert kth_successor(find(root, 2), 3).val == 5
assert kth_successor(find(root, 2), 4).val == 6
assert kth_successor(find(root, 2), 5) is None
assert kth_successor(find(root, 6), 1) is None

def oracle_predecessor(node):
    rt = node
    while rt.parent is not None:
        rt = rt.parent
    nodes = sorted(all_nodes(rt), key=lambda x: x.val)
    i = bisect.bisect_left([x.val for x in nodes], node.val) - 1
    return nodes[i] if i >= 0 else None

rng = random.Random(5)
for _ in range(100):
    rt = random_bst(rng, rng.randint(1, 40))
    for nd in all_nodes(rt):
        assert predecessor(nd) is oracle_predecessor(nd)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template successor với con trỏ `parent`:**

```python
class Solution:
    def inorderSuccessor(self, node):
        if node.right:                         # (a) có con phải
            cur = node.right
            while cur.left:
                cur = cur.left
            return cur
        cur = node                             # (b) leo lên tới khi là con trái
        while cur.parent and cur is cur.parent.right:
            cur = cur.parent
        return cur.parent                      # None nếu đã tới gốc
```

**Checklist trước khi submit:**

- [ ] Có xử lý **cả hai** trường hợp: có con phải / không có con phải?
- [ ] Trường hợp có con phải: đi **trái hết mức** sau khi sang phải một bước?
- [ ] Vòng leo: kiểm tra `cur.parent` **trước** `cur.parent.right` (tránh `None.right`)?
- [ ] Trả về **nút** (`Node`), không phải `.val`?
- [ ] Trả `None` khi `node` là nút lớn nhất?
- [ ] Đã thử: cây một nút, nút lớn nhất, nút nhỏ nhất, nút là con phải của con trái...?
- [ ] Đã so sánh với oracle trên cây ngẫu nhiên và mọi nút?